# 12-bob. 12-BOB. XAVFSIZLIK OPERATSIYALARI MARKAZIDA SI
## Amaliy misol: jurnal shablonlari va korrelyatsiya qoidasi

[![Colab'da ochish](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mrmsi/si-kiberxavfsizlik/blob/main/12-bob/12-1_jurnal-shablonlari-va-korrelyatsiya-qoidasi.ipynb)

*"Sun'iy intellektni kiberxavfsizlikda qo'llash" darsligi, 12-bob.*

Quyidagi dasturda bir kunlik SSH xizmati jurnali sun'iy ravishda yaratiladi. Unda xodimlarning ish vaqtidagi odatiy kirish va chiqishlari bilan birga, tungi soat 3 da tashqi manzildan amalga oshirilgan parol tanlash hujumi va uning oxirida muvaffaqiyatli kirish mavjud. Dastur avval o'zgaruvchan qismlarni (IP manzillar, sonlar, foydalanuvchi nomlari) maxsus belgilar bilan almashtirib, yozuvlarni shablonlarga ajratadi. So'ngra korrelyatsiya qoidasini qo'llaydi: bitta manzildan 20 dan ortiq muvaffaqiyatsiz urinishdan keyin muvaffaqiyatli kirish. Misolda tashqi IP manzil sifatida o'quv misollari uchun ajratilgan manzil ishlatilgan. Dastur amaliy mashg'ulotning A12 mavzusiga mos keladi.

**Ishga tushirish:** yuqoridagi menyudan *Runtime → Run all* (*Ishga tushirish → Hammasini bajarish*) ni tanlang yoki kataklarni birma-bir Shift+Enter bilan bajaring. Barcha ma'lumotlar dasturning o'zida yaratiladi, qo'shimcha fayl yuklash shart emas.

In [1]:
import random
import re
from collections import Counter, defaultdict

tasodif = random.Random(5)
foydalanuvchilar = ["aziz", "dilnoza", "jasur", "malika", "sardor"]
jurnal = []

In [2]:
# 1. Bir kunlik odatiy SSH jurnali (sun'iy ma'lumot)
for _ in range(3000):
    soat = f"{tasodif.randint(8, 19):02d}:{tasodif.randint(0, 59):02d}:{tasodif.randint(0, 59):02d}"
    ip = f"10.0.{tasodif.randint(1, 5)}.{tasodif.randint(2, 250)}"
    f = tasodif.choice(foydalanuvchilar)
    port = tasodif.randint(40000, 65000)
    hodisa = tasodif.choices([
        f"Accepted password for {f} from {ip} port {port} ssh2",
        f"Failed password for {f} from {ip} port {port} ssh2",
        f"session opened for user {f}",
        f"session closed for user {f}",
        f"Disconnected from user {f} {ip} port {port}",
    ], weights=[20, 2, 20, 20, 20])[0]
    jurnal.append((soat, hodisa))

In [3]:
# 2. Tunda tashqi manzildan parol tanlash va muvaffaqiyatli kirish
for i in range(400):
    jurnal.append((f"03:{i // 60:02d}:{i % 60:02d}",
                   f"Failed password for invalid user {tasodif.choice(['admin', 'test', 'oracle', 'root'])} "
                   f"from 203.0.113.77 port {tasodif.randint(40000, 65000)} ssh2"))
jurnal.append(("03:07:12", "Accepted password for root from 203.0.113.77 port 51515 ssh2"))
jurnal.sort()

In [4]:
# 3. Shablonlarni ajratish: o'zgaruvchan qismlarni maxsus belgilar bilan almashtirish
def shablon(xabar):
    xabar = re.sub(r"\b\d{1,3}(\.\d{1,3}){3}\b", "<IP>", xabar)
    xabar = re.sub(r"\b\d+\b", "<SON>", xabar)
    xabar = re.sub(r"(for|user) (invalid user |user )?(\w+)", lambda m: f"{m.group(1)} {m.group(2) or ''}<FOYD>", xabar)
    return xabar

shablonlar = Counter(shablon(x) for _, x in jurnal)
print(f"Jurnal yozuvlari: {len(jurnal)}, shablonlar soni: {len(shablonlar)}\n")
for s, n in shablonlar.most_common():
    print(f"{n:>6}  {s}")

Jurnal yozuvlari: 3401, shablonlar soni: 6

   753  session opened for user <FOYD>
   748  Accepted password for <FOYD> from <IP> port <SON> ssh2
   721  Disconnected from user <FOYD> <IP> port <SON>
   718  session closed for user <FOYD>
   400  Failed password for invalid user <FOYD> from <IP> port <SON> ssh2
    61  Failed password for <FOYD> from <IP> port <SON> ssh2


In [5]:
# 4. Korrelyatsiya qoidasi: bitta manzildan 20+ muvaffaqiyatsiz urinishdan keyin muvaffaqiyatli kirish
xatolar = defaultdict(int)
print("\nOgohlantirishlar:")
for soat, x in jurnal:
    ip = re.search(r"\d{1,3}(\.\d{1,3}){3}", x)
    if not ip:
        continue
    if x.startswith("Failed"):
        xatolar[ip.group()] += 1
    elif x.startswith("Accepted") and xatolar[ip.group()] >= 20:
        print(f"  {soat} {ip.group()}: {xatolar[ip.group()]} ta muvaffaqiyatsiz urinishdan keyin kirish -> {x}")


Ogohlantirishlar:
  03:07:12 203.0.113.77: 400 ta muvaffaqiyatsiz urinishdan keyin kirish -> Accepted password for root from 203.0.113.77 port 51515 ssh2


## Kitobdagi natija

Dasturni ishga tushirganingizda quyidagiga o'xshash natija chiqishi kerak (tasodifiy sonlar generatori urug'i qat'iy belgilangani uchun natija odatda aynan bir xil bo'ladi; faqat bajarilish vaqti va har safar yangi yaratiladigan tasodifiy kalitlar farq qilishi mumkin):

```
Jurnal yozuvlari: 3401, shablonlar soni: 6

   753  session opened for user <FOYD>
   748  Accepted password for <FOYD> from <IP> port <SON> ssh2
   721  Disconnected from user <FOYD> <IP> port <SON>
   718  session closed for user <FOYD>
   400  Failed password for invalid user <FOYD> from <IP> port <SON> ssh2
    61  Failed password for <FOYD> from <IP> port <SON> ssh2

Ogohlantirishlar:
  03:07:12 203.0.113.77: 400 ta muvaffaqiyatsiz urinishdan keyin kirish -> Accepted password for root from 203.0.113.77 port 51515 ssh2
```

## O'zingiz sinab ko'ring

Parametrlarni (ma'lumotlar hajmi, chegara qiymati, tasodifiy son urug'i) o'zgartirib, natija qanday o'zgarishini kuzating. Kitobdagi 12-bobning B darajadagi topshiriqlari aynan shu daftar asosida bajariladi.